# Benchmarking and Evaluating Qwen3.5-9B on GPU

In this notebook, you'll learn how to:
1. Run a real **vLLM serving benchmark** against the GPU server from L6
2. Interpret throughput, TTFT, TPOT, ITL, and tail latency
3. Evaluate model quality with **lm-evaluation-harness**
4. Combine serving and quality evidence into a deployment decision

## Setup

This notebook reuses the **Qwen3.5-9B CUDA server** started by L6 at `http://localhost:8000`. If that container is still running, setup is immediate and the downloaded model remains cached.

| Dimension | Question | Tool |
|:--|:--|:--|
| **Performance** | How fast and efficiently does the deployment serve requests? | `vllm bench serve` |
| **Quality** | How well does the model perform on a standardized task? | lm-evaluation-harness |

Select the same project `.venv` kernel used for L6 before running the code cells.

Let's send a quick test request to confirm everything is working.

In [1]:
import json, os, statistics, subprocess, sys, time
from pathlib import Path

import requests

VLLM_URL = os.getenv("VLLM_URL", "http://localhost:8000").rstrip("/")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "Qwen/Qwen3.5-9B")
VLLM_IMAGE = os.getenv("VLLM_IMAGE", "vllm/vllm-openai:latest")
OUTPUT_DIR = Path("outputs/l7_qwen35")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    response = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
    response.raise_for_status()
    MODEL = response.json()["data"][0]["id"]
except (requests.RequestException, KeyError, IndexError) as exc:
    raise RuntimeError(
        "The vLLM server is not ready. Run the first code cell in L6, wait for "
        "Qwen3.5-9B to load, then rerun this cell."
    ) from exc

print(f"Connected to {VLLM_URL} - served model: {MODEL}")
print(f"Tokenizer/config source: {MODEL_SOURCE}")

Connected to http://localhost:8000 - served model: qwen3.5-9b
Tokenizer/config source: Qwen/Qwen3.5-9B


In [3]:
from openai import OpenAI

client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")
response = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": "What is model quantization in one sentence?",
    }],
    max_tokens=60,
    temperature=0.7,
    top_p=0.8,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False},
    },
)
print(f"{MODEL}: {response.choices[0].message.content.strip()}")

qwen3.5-9b: Model quantization is the process of reducing the numerical precision of a machine learning model's weights and activations (e.g., from 32-bit floating-point to 8-bit integers) to significantly decrease memory usage and accelerate inference with minimal impact on accuracy.


## Benchmarking with vLLM

The vLLM image includes `vllm bench serve`, the same benchmark client used by this repository's full BF16/GPTQ/AWQ/FP8 experiments.

| Metric | What it measures |
|:--|:--|
| **TTFT** | Time to first token: prefill and queue responsiveness |
| **TPOT** | Time per output token after the first token |
| **ITL** | Inter-token latency: streaming smoothness |
| **Throughput** | Completed requests and generated tokens per second |

This instructional run uses **20 prompts**, 128 input tokens, 64 output tokens, and concurrency 4. It is a smoke benchmark, not a statistically strong p99 measurement. The larger benchmark lab uses 200 prompts, three seeds, and several concurrency levels.

In [4]:
BENCHMARK_FILE = OUTPUT_DIR / "qwen35_smoke.json"
if BENCHMARK_FILE.exists():
    BENCHMARK_FILE.unlink()

In [5]:
benchmark_command = [
    "docker", "run", "--rm", "--network", "host",
    "-v", f"{OUTPUT_DIR.resolve()}:/results",
    "-v", f"{Path.home() / '.cache' / 'huggingface'}:/root/.cache/huggingface",
    "--entrypoint", "", VLLM_IMAGE,
    "vllm", "bench", "serve",
    "--backend", "vllm",
    "--base-url", VLLM_URL,
    "--endpoint", "/v1/completions",
    "--model", MODEL,
    "--tokenizer", MODEL_SOURCE,
    "--dataset-name", "random",
    "--random-input-len", "128",
    "--random-output-len", "64",
    "--num-prompts", "20",
    "--request-rate", "inf",
    "--max-concurrency", "4",
    "--ignore-eos", "--seed", "42",
    "--save-result", "--save-detailed",
    "--result-dir", "/results",
    "--result-filename", BENCHMARK_FILE.name,
]

print("Running the GPU serving smoke benchmark...")
result = subprocess.run(
    benchmark_command, capture_output=True, text=True, timeout=900
)
if result.returncode != 0:
    raise RuntimeError(
        f"vLLM benchmark failed ({result.returncode}).\n"
        f"STDOUT:\n{result.stdout[-2000:]}\nSTDERR:\n{result.stderr[-4000:]}"
    )
if not BENCHMARK_FILE.exists():
    raise RuntimeError("Benchmark completed without producing the expected JSON file.")

print(f"Benchmark complete: {BENCHMARK_FILE}")
print(result.stdout[-1500:])

Running the GPU serving smoke benchmark...
Benchmark complete: outputs/l7_qwen35/qwen35_smoke.json
point ready check.
Starting main benchmark run...
Traffic request rate: inf
Burstiness factor: 1.0 (Poisson process)
Maximum request concurrency: 4
tip: install termplotlib and gnuplot to plot the metrics
============ Serving Benchmark Result ============
Successful requests:                     20        
Failed requests:                         0         
Maximum request concurrency:             4         
Benchmark duration (s):                  26.76     
Total input tokens:                      2560      
Total generated tokens:                  1280      
Request throughput (req/s):              0.75      
Output token throughput (tok/s):         47.83     
Peak output token throughput (tok/s):    52.00     
Peak concurrent requests:                8.00      
Total token throughput (tok/s):          143.49    
---------------Time to First Token----------------
Mean TTFT (ms):       

## Interpreting Benchmark Results

`vllm bench serve` writes both summary statistics and per-request samples. The next cell reads the run-specific result file, preventing a failed run from accidentally displaying stale metrics.

In [6]:
with BENCHMARK_FILE.open() as file:
    benchmark = json.load(file)

print(f"Model: {benchmark['model_id']}")
print(
    f"Requests: {benchmark['completed']} completed, "
    f"{benchmark['failed']} failed | "
    f"Concurrency: {benchmark['max_concurrency']}\n"
)

print(f"{'Metric':<24} {'Value':>12}")
print("-" * 38)
summary_metrics = [
    ("Request throughput", benchmark["request_throughput"], "req/s"),
    ("Output throughput", benchmark["output_throughput"], "tok/s"),
    ("Total token throughput", benchmark["total_token_throughput"], "tok/s"),
    ("Mean TTFT", benchmark["mean_ttft_ms"], "ms"),
    ("Median TTFT", benchmark["median_ttft_ms"], "ms"),
    ("P99 TTFT", benchmark["p99_ttft_ms"], "ms"),
    ("Median TPOT", benchmark["median_tpot_ms"], "ms"),
    ("P99 TPOT", benchmark["p99_tpot_ms"], "ms"),
    ("Median ITL", benchmark["median_itl_ms"], "ms"),
    ("P99 ITL", benchmark["p99_itl_ms"], "ms"),
]
for label, value, unit in summary_metrics:
    print(f"{label:<24} {value:>9.2f} {unit}")

Model: qwen3.5-9b
Requests: 20 completed, 0 failed | Concurrency: 4

Metric                          Value
--------------------------------------
Request throughput            0.75 req/s
Output throughput            47.83 tok/s
Total token throughput      143.49 tok/s
Mean TTFT                   222.84 ms
Median TTFT                 237.64 ms
P99 TTFT                    251.30 ms
Median TPOT                  81.23 ms
P99 TPOT                     82.58 ms
Median ITL                   81.01 ms
P99 ITL                      87.29 ms


**Why Percentiles Matter**

Averages hide outliers. A system with 100 ms average latency but 2-second p99 means **1 in 100 requests** takes at least 20 times as long as the average.

| Percentile | Meaning |
|:--|:--|
| **p50 (median)** | Half of requests are faster |
| **p95** | 95% are faster; the remaining 5% form the tail |
| **p99** | 99% are faster; only 1 in 100 is slower |

`vllm bench serve` reports percentile latency directly. This 20-request smoke run is useful for catching regressions, but it is too small for a stable p99. Use the larger multi-seed runs from the benchmark lab for deployment decisions.

## Evaluating Model Quality with lm_eval

Performance benchmarks tell you how fast a deployment is, but not whether it gives **good answers**.

[lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) measures task performance on standardized datasets.

| | `vllm bench serve` | lm-evaluation-harness |
|:--|:--|:--|
| **Measures** | Serving speed, latency, throughput | Task accuracy and quality |
| **Target** | A running inference server | A model served through an evaluation backend |
| **Question** | "How well does this deployment perform?" | "How well does this model answer?" |

The evaluator targets the same vLLM server through its OpenAI-compatible **completions** endpoint. Hellaswag is multiple choice, so the harness scores each candidate continuation using token log probabilities.

> This lesson uses an explicit **0-shot** configuration and only **20 validation examples**. Treat the result as an integration smoke test, not a publishable quality estimate; run the full task with fixed harness and model revisions before comparing models.

In [2]:
import os
from pathlib import Path

EVAL_HF_HOME = Path("outputs/l7_qwen35/hf_cache").resolve()
EVAL_HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(EVAL_HF_HOME)
os.environ["HF_DATASETS_CACHE"] = str(EVAL_HF_HOME / "datasets")
os.environ.setdefault("OPENAI_API_KEY", "unused")

import lm_eval

TASK = "hellaswag"
EVAL_LIMIT = 20
NUM_FEWSHOT = 0
print(
    f"Running lm_eval on {MODEL} via vLLM "
    f"({TASK}, {NUM_FEWSHOT}-shot, {EVAL_LIMIT} examples)...\n"
 )

results = lm_eval.simple_evaluate(
    model="local-completions",
    model_args=(
        f"model={MODEL},"
        f"base_url={VLLM_URL}/v1/completions,"
        "tokenized_requests=False,"
        f"tokenizer={MODEL_SOURCE},"
        "num_concurrent=1"
    ),
    tasks=[TASK],
    num_fewshot=NUM_FEWSHOT,
    limit=EVAL_LIMIT,
)

Running lm_eval on qwen3.5-9b via vLLM (hellaswag, 0-shot, 20 examples)...



config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10042 [00:00<?, ? examples/s]

Map:   0%|          | 0/39905 [00:00<?, ? examples/s]

Map:   0%|          | 0/10042 [00:00<?, ? examples/s]

Overwriting default num_fewshot of hellaswag from None to 0
Requesting API: 100%|██████████| 80/80 [00:09<00:00,  8.75it/s]


In [3]:
task_results = results["results"][TASK]

print(f"Model: {MODEL}")
print(
    f"Task: {TASK} | Few-shot examples: {NUM_FEWSHOT} "
    f"| Evaluated examples: {EVAL_LIMIT}\n"
 )
for metric, value in task_results.items():
    if isinstance(value, (int, float)):
        print(f"  {metric}: {value:.4f}")

Model: qwen3.5-9b
Task: hellaswag | Few-shot examples: 0 | Evaluated examples: 20

  sample_len: 20.0000
  acc,none: 0.4500
  acc_stderr,none: 0.1141
  acc_norm,none: 0.6500
  acc_norm_stderr,none: 0.1094


## Comparing with Earlier Compression Labs

The earlier compression labs produced BF16, AWQ, GPTQ, and FP8 serving artifacts and benchmark reports. Reuse that workflow here by serving each candidate with the same vLLM settings, then rerunning both sections of L7.

| Keep fixed | Change per candidate |
|:--|:--|
| Prompt dataset, input/output lengths, concurrency, seed | Model path and served-model name |
| vLLM image, GPU, memory utilization, benchmark command | Quantization format |
| lm_eval task, split, few-shot count, harness revision | Candidate checkpoint |

Record one row per candidate:

| Candidate | Size (GiB) | Request/s | Output tok/s | Median TTFT (ms) | p99 TPOT (ms) | Hellaswag `acc_norm` |
|:--|--:|--:|--:|--:|--:|--:|
| Qwen3.5-9B baseline | Measure | Measure | Measure | Measure | Measure | Measure |
| Matching-family AWQ/GPTQ/FP8 | Measure | Measure | Measure | Measure | Measure | Measure |

Do not compare the Qwen3.5-9B result directly with the repository's existing Qwen3-8B compressed artifacts: changing both the base model and quantization format makes the cause of any difference ambiguous. Create or obtain compressed checkpoints from the **same Qwen3.5-9B base revision** for an apples-to-apples deployment study.

For each metric, calculate quality retention and performance gain against the baseline:

$$\text{quality retention} = \frac{\text{candidate accuracy}}{\text{baseline accuracy}} \times 100\%$$

$$\text{throughput gain} = \frac{\text{candidate throughput}}{\text{baseline throughput}}$$

## Making the Decision

You now have three sources of evidence:

| Source | What it tells you |
|:--|:--|
| **`vllm bench serve`** | Whether the deployment meets latency and throughput objectives |
| **lm-evaluation-harness** | Whether model quality is retained on a task you control |
| **Earlier compression labs** | How model size and quantization format affect memory and serving behavior |

Choose a candidate only after setting acceptance thresholds for the use case, such as maximum p99 TTFT, minimum output throughput, GPU-memory budget, and minimum quality retention. The 20-request and 20-example runs in this notebook verify the pipeline; larger repeated runs provide decision-grade evidence.

For the current Qwen3.5-9B baseline, keep the benchmark JSON and lm_eval configuration as the control. Evaluate matching-family AWQ, GPTQ, or FP8 checkpoints with exactly the same settings, then select the smallest candidate that still meets both the service-level objectives and the quality threshold.

## Summary


In this notebook, you:

- Reused the **Qwen3.5-9B GPU server** from L6
- Ran `vllm bench serve` and measured TTFT, TPOT, ITL, and throughput
- Evaluated 0-shot Hellaswag quality through the live OpenAI-compatible endpoint
- Isolated notebook evaluation data from the Docker-owned model cache
- Defined an apples-to-apples comparison with the BF16/AWQ/GPTQ/FP8 workflow from the earlier labs
- Combined performance, memory, and quality evidence into a deployment decision

## Resources

- [vLLM serving benchmark documentation](https://docs.vllm.ai/en/latest/cli/bench/serve.html)
- [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
- [Qwen3.5-9B model card](https://huggingface.co/Qwen/Qwen3.5-9B)
- Repository benchmark workflow: `Compressor/vllm_bench/`